# Tema 3 – Rețele Bayesiene
## Exercițiul 3 – Jocul dintre J0 și J1

### Descrierea problemei

Jocul are două runde:

1. Se aruncă o monedă normală pentru a decide cine începe: `J0` sau `J1`.
2. Jucătorul care începe aruncă un zar normal și obține valoarea `n`.
3. Celălalt jucător aruncă propria monedă de `2*n` ori. Notăm cu `m` numărul de steme obținute.
4. Jucătorul din prima rundă câștigă dacă `n >= m`; altfel câștigă jucătorul din a doua rundă.

Moneda lui `J0` este normală, deci probabilitatea de stemă este `1/2`. Moneda lui `J1` este măsluită, cu probabilitatea de stemă `4/7`. Ambele zaruri sunt normale.

Vom simula 10.000 de jocuri, apoi vom construi o rețea Bayesiană și vom calcula cine este mai probabil să fi început, știind că în runda a doua s-a obținut exact o stemă.

## a) Simularea a 10.000 de jocuri

Folosim `random` pentru a simula moneda care stabilește cine începe, zarul normal și aruncările repetate ale monedei celui de-al doilea jucător.

Dacă jucătorul din prima rundă obține `n` la zar, al doilea jucător aruncă moneda de `2*n` ori. Primul câștigă când `n >= m`; în caz contrar, câștigă al doilea.

In [ ]:
import random

random.seed(42)  # Pentru rezultate reproductibile

N = 10_000
victorii_J0 = 0
victorii_J1 = 0

for _ in range(N):
    # Monedă normală: decide cine începe
    incepe_J0 = random.choice([True, False])
    primul = "J0" if incepe_J0 else "J1"
    al_doilea = "J1" if incepe_J0 else "J0"

    # Zar normal al jucătorului care începe
    n = random.randint(1, 6)

    # Moneda celui de-al doilea jucător
    p_stema = 0.5 if al_doilea == "J0" else 4 / 7
    m = sum(random.random() < p_stema for _ in range(2 * n))

    # Stabilim câștigătorul
    castigator = primul if n >= m else al_doilea

    if castigator == "J0":
        victorii_J0 += 1
    else:
        victorii_J1 += 1

print(f"Număr jocuri: {N}")
print(f"Victorii J0: {victorii_J0} ({victorii_J0 / N:.2%})")
print(f"Victorii J1: {victorii_J1} ({victorii_J1 / N:.2%})")
print("Jucătorul cu cele mai multe victorii:",
      "J0" if victorii_J0 > victorii_J1 else "J1")

### Explicația simulării

- `random.choice([True, False])` simulează moneda normală care stabilește cine începe.
- `random.randint(1, 6)` simulează zarul normal și produce un număr între 1 și 6.
- `p_stema` este `1/2` dacă moneda folosită aparține lui `J0`, respectiv `4/7` dacă aparține lui `J1`.
- `sum(random.random() < p_stema for _ in range(2*n))` simulează cele `2*n` aruncări și numără steme.
- Dacă `n >= m`, câștigă primul jucător; altfel, câștigă al doilea.
- Rezultatele sunt estimări ale probabilităților de câștig, deoarece simulăm un număr finit de jocuri.

## b) Definirea unei rețele Bayesiene

Pentru a modela jocul, folosim variabilele:

- `S`: cine începe (`0` = J0, `1` = J1);
- `N`: rezultatul zarului primului jucător (1–6);
- `M`: numărul de steme din runda a doua (0–12).

Structura este `S → M` și `N → M`. Variabila `M` depinde de cine începe, deoarece acest lucru determină cine aruncă moneda în runda a doua, și de `N`, deoarece numărul de aruncări este `2*N`.

`S` și `N` sunt independente în model: moneda care decide cine începe și zarul normal nu influențează unul rezultatul celuilalt.

In [ ]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD

# S=0: J0 începe; S=1: J1 începe
# N este indexat de la 0 la 5 în rețea, reprezentând valorile reale 1..6.
# M este indexat de la 0 la 12 și reprezintă numărul de steme.

model_joc = DiscreteBayesianNetwork([
    ("S", "M"),
    ("N", "M")
])

cpd_s = TabularCPD(
    variable="S",
    variable_card=2,
    values=[[0.5], [0.5]]
)

cpd_n = TabularCPD(
    variable="N",
    variable_card=6,
    values=[[1/6], [1/6], [1/6], [1/6], [1/6], [1/6]]
)

# Distribuția lui M condiționată de S și N.
# Coloanele sunt ordonate după S apoi N:
# pentru fiecare S, N ia pe rând valorile 0,1,2,3,4,5 (adică n=1..6).
# Dacă J0 începe (S=0), J1 aruncă moneda cu p(stemă)=4/7.
# Dacă J1 începe (S=1), J0 aruncă moneda normală cu p(stemă)=1/2.
from math import comb

values_m = []
for m in range(13):
    row = []
    for s in range(2):
        for n_index in range(6):
            n_real = n_index + 1
            p = 4/7 if s == 0 else 1/2
            if m <= 2 * n_real:
                prob = comb(2 * n_real, m) * (p ** m) * ((1-p) ** (2*n_real-m))
            else:
                prob = 0.0
            row.append(prob)
    values_m.append(row)

cpd_m = TabularCPD(
    variable="M",
    variable_card=13,
    values=values_m,
    evidence=["S", "N"],
    evidence_card=[2, 6]
)

model_joc.add_cpds(cpd_s, cpd_n, cpd_m)
print("Model valid:", model_joc.check_model())

### Explicația rețelei

`S` și `N` au distribuții uniforme: fiecare jucător are probabilitatea `1/2` de a începe, iar fiecare rezultat al zarului are probabilitatea `1/6`.

Pentru `M`, folosim distribuția binomială. Dacă moneda are probabilitatea `p` de stemă și este aruncată de `2*n` ori, atunci:

\[
P(M=m\mid S,N=n)=\binom{2n}{m}p^m(1-p)^{2n-m}.
\]

În această rețea, `p=4/7` când `J0` începe, deoarece `J1` aruncă moneda măsluită; `p=1/2` când `J1` începe, deoarece `J0` aruncă moneda normală.

## c) Cine este mai probabil să fi început, dacă s-a obținut o singură stemă?

Observăm `M=1` și calculăm:

\[
P(S=0\mid M=1),\qquad P(S=1\mid M=1).
\]

În rețea, valorile lui `N` sunt indexate de la 0 la 5 pentru a reprezenta rezultatele reale ale zarului de la 1 la 6. Prin urmare, dovezile pentru inferență sunt `M=1`.

In [ ]:
from pgmpy.inference import VariableElimination

infer_joc = VariableElimination(model_joc)
posterior_s = infer_joc.query(
    variables=["S"],
    evidence={"M": 1}
)

print(posterior_s)
print(f"P(J0 începe | M=1) = {float(posterior_s.values[0]):.6f}")
print(f"P(J1 începe | M=1) = {float(posterior_s.values[1]):.6f}")

if posterior_s.values[0] > posterior_s.values[1]:
    print("Este mai probabil ca J0 să fi început.")
else:
    print("Este mai probabil ca J1 să fi început.")

### Interpretarea rezultatului

- `S=0` înseamnă că J0 a început.
- `S=1` înseamnă că J1 a început.
- Inferența folosește faptul că s-a observat exact o stemă în runda a doua.

Comparăm cele două probabilități posterioare. Cea mai mare indică jucătorul care este mai probabil să fi început jocul.

## Concluzie

Am estimat șansele de câștig prin simularea a 10.000 de jocuri. Apoi am construit o rețea Bayesiană în care numărul de steme depinde de jucătorul care începe și de rezultatul zarului.

Distribuția posterioară a variabilei `S`, condiționată de `M=1`, ne permite să stabilim cine este mai probabil să fi început jocul atunci când în runda a doua s-a obținut o singură stemă.